<a href="https://colab.research.google.com/github/NikitaIvagin/ml-portfolio/blob/main/neuroevolution/genetic_cnn_mlp/Genetic_CNN_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Гиперпараметрическая оптимизация CNN и MLP с помощью генетического алгоритма на CIFAR-10

### 1. Загрузка и предобработка данных CIFAR-10

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import numpy as np
import random

# Загрузка данных CIFAR-10
(X_train_full, y_train_full), (X_test, y_test) = cifar10.load_data()

# Нормализация данных
X_train_full = X_train_full.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Преобразование меток в one-hot кодирование
y_train_full = to_categorical(y_train_full, 10)
y_test = to_categorical(y_test, 10)

# Разделение обучающей выборки на тренировочную и валидационную
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)

print(f"Форма X_train: {X_train.shape}")
print(f"Форма y_train: {y_train.shape}")
print(f"Форма X_val: {X_val.shape}")
print(f"Форма y_val: {y_val.shape}")
print(f"Форма X_test: {X_test.shape}")
print(f"Форма y_test: {y_test.shape}")

# Настройка стратегии распределения для GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Используем MirroredStrategy для поддержки одного или нескольких GPU
        strategy = tf.distribute.MirroredStrategy()
        print(f'Обнаружено {len(gpus)} GPU: {gpus}')
    except RuntimeError as e:
        print(e)
        strategy = tf.distribute.get_strategy()
else:
    strategy = tf.distribute.get_strategy()
    print('GPU не обнаружен, используется CPU.')


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Форма X_train: (40000, 32, 32, 3)
Форма y_train: (40000, 10)
Форма X_val: (10000, 32, 32, 3)
Форма y_val: (10000, 10)
Форма X_test: (10000, 32, 32, 3)
Форма y_test: (10000, 10)
GPU не обнаружен, используется CPU.


### 2. Определение архитектур моделей

Теперь определим функции для создания моделей MLP и CNN. Эти функции будут принимать гиперпараметры, найденные генетическим алгоритмом.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.optimizers import Adam

# Функция для создания MLP модели
def build_mlp_model(input_shape, num_classes, hyperparameters):
    with strategy.scope(): # Создаем модель внутри области стратегии
        model = Sequential([
            Flatten(input_shape=input_shape) # Преобразуем 2D изображения в 1D вектор
        ])

        num_layers = hyperparameters['num_layers']
        num_neurons = hyperparameters['num_neurons']
        activation = hyperparameters['activation']

        for _ in range(num_layers):
            model.add(Dense(num_neurons, activation=activation))
            model.add(Dropout(0.2))

        model.add(Dense(num_classes, activation='softmax'))

        model.compile(
            optimizer=Adam(learning_rate=hyperparameters['learning_rate']),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
    return model

# Функция для создания CNN модели
def build_cnn_model(input_shape, num_classes, hyperparameters):
    with strategy.scope(): # Создаем модель внутри области стратегии
        model = Sequential()

        num_conv_layers = hyperparameters['num_conv_layers']
        filters_per_layer = hyperparameters['filters_per_layer']
        kernel_size = hyperparameters['kernel_size']
        activation = hyperparameters['activation']

        # Первый сверточный слой
        model.add(Conv2D(filters_per_layer[0], kernel_size, activation=activation, input_shape=input_shape, padding='same'))
        model.add(MaxPooling2D((2, 2)))

        # Дополнительные сверточные слои
        for i in range(1, num_conv_layers):
            model.add(Conv2D(filters_per_layer[i], kernel_size, activation=activation, padding='same'))
            model.add(MaxPooling2D((2, 2)))

        model.add(Flatten())

        num_dense_layers = hyperparameters['num_dense_layers']
        num_dense_neurons = hyperparameters['num_dense_neurons']

        for _ in range(num_dense_layers):
            model.add(Dense(num_dense_neurons, activation=activation))
            model.add(Dropout(0.3))

        model.add(Dense(num_classes, activation='softmax'))

        model.compile(
            optimizer=Adam(learning_rate=hyperparameters['learning_rate']),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
    return model

print("Функции build_mlp_model и build_cnn_model определены.")


Функции build_mlp_model и build_cnn_model определены.


### 3. Генетический алгоритм для оптимизации MLP

In [ ]:
# Определение пространства поиска гиперпараметров для MLP
mlp_hyperparameter_space = {
    'learning_rate': (0.0001, 0.01), # Диапазон для learning_rate
    'num_layers': (1, 3),          # Количество скрытых слоев
    'num_neurons': (64, 256),      # Количество нейронов в каждом скрытом слое
    'activation': ['relu', 'tanh'] # Функции активации
}

# Функция инициализации особи для MLP
def initialize_mlp_individual():
    lr = random.uniform(mlp_hyperparameter_space['learning_rate'][0],
                        mlp_hyperparameter_space['learning_rate'][1])
    num_layers = random.randint(mlp_hyperparameter_space['num_layers'][0],
                               mlp_hyperparameter_space['num_layers'][1])
    num_neurons = random.randint(mlp_hyperparameter_space['num_neurons'][0],
                                mlp_hyperparameter_space['num_neurons'][1])
    activation = random.choice(mlp_hyperparameter_space['activation'])
    return {'learning_rate': lr, 'num_layers': num_layers, 'num_neurons': num_neurons, 'activation': activation}

# Функция приспособленности для MLP (оценка модели)
def fitness_function_mlp(individual, X_train, y_train, X_val, y_val, input_shape, num_classes, epochs=5):
    hyperparameters = individual
    model = build_mlp_model(input_shape, num_classes, hyperparameters)

    # Обучение модели
    history = model.fit(X_train, y_train, epochs=epochs, validation_data=(X_val, y_val), verbose=0)

    # Возвращаем максимальную точность на валидационном наборе
    return max(history.history['val_accuracy'])

# Универсальные функции генетического алгоритма
def select_parents(population, fitness_scores, num_parents):
    # Турнирный отбор
    selected_parents = []
    for _ in range(num_parents):
        tournament_size = 3
        competitors_indices = random.sample(range(len(population)), tournament_size)
        winner_index = competitors_indices[np.argmax([fitness_scores[i] for i in competitors_indices])]
        selected_parents.append(population[winner_index])
    return selected_parents

def crossover_mlp(parent1, parent2):
    child1 = parent1.copy()
    child2 = parent2.copy()

    # Случайная точка кроссовера для каждого гиперпараметра
    for key in parent1.keys():
        if random.random() < 0.5: # 50% шанс обмена
            child1[key], child2[key] = parent2[key], parent1[key]

    return child1, child2

def mutate_mlp(individual, mutation_rate):
    mutated_individual = individual.copy()
    for key, (min_val, max_val) in mlp_hyperparameter_space.items():
        if random.random() < mutation_rate:
            if key == 'learning_rate':
                mutated_individual[key] = random.uniform(min_val, max_val)
            elif key == 'num_layers':
                mutated_individual[key] = random.randint(min_val, max_val)
            elif key == 'num_neurons':
                mutated_individual[key] = random.randint(min_val, max_val)
            elif key == 'activation':
                mutated_individual[key] = random.choice(mlp_hyperparameter_space['activation'])
    return mutated_individual

def run_genetic_algorithm(
    initialize_individual_func,
    fitness_func,
    crossover_func,
    mutate_func,
    population_size,
    num_generations,
    mutation_rate,
    **fitness_func_args
):
    population = [initialize_individual_func() for _ in range(population_size)]
    best_individual = None
    best_fitness = -1

    for generation in range(num_generations):
        print(f"\n--- Поколение {generation + 1}/{num_generations} ---")
        fitness_scores = []
        for i, individual in enumerate(population):
            print(f"Оцениваем особь {i+1}/{population_size}: {individual}")
            fitness = fitness_func(individual, **fitness_func_args)
            fitness_scores.append(fitness)
            if fitness > best_fitness:
                best_fitness = fitness
                best_individual = individual
                print(f"Новый лучший результат: {best_fitness:.4f} с параметрами: {best_individual}")

        # Отбор
        parents = select_parents(population, fitness_scores, population_size // 2)

        # Создание нового поколения
        next_population = []
        for i in range(0, len(parents), 2):
            if i + 1 < len(parents): # Убедиться, что есть вторая пара
                parent1 = parents[i]
                parent2 = parents[i+1]
                child1, child2 = crossover_func(parent1, parent2)
                next_population.append(mutate_func(child1, mutation_rate))
                next_population.append(mutate_func(child2, mutation_rate))
            else: # Если остался один родитель, просто добавить его мутировавшую версию
                next_population.append(mutate_func(parents[i], mutation_rate))

        # Если next_population получилась меньше, чем population_size из-за округлений, добить случайными
        while len(next_population) < population_size:
            next_population.append(initialize_individual_func())

        population = next_population

    return best_individual, best_fitness

# Параметры генетического алгоритма
population_size = 5
num_generations = 3
mutation_rate = 0.1

# Запуск ГА для MLP
print("Запуск генетического алгоритма для MLP...")
best_mlp_params, best_mlp_accuracy = run_genetic_algorithm(
    initialize_mlp_individual,
    fitness_function_mlp,
    crossover_mlp,
    mutate_mlp,
    population_size,
    num_generations,
    mutation_rate,
    X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
    input_shape=X_train.shape[1:], num_classes=y_train.shape[1]
)

print("\n--- Результаты оптимизации MLP ---")
print(f"Лучшие гиперпараметры MLP: {best_mlp_params}")
print(f"Лучшая точность MLP на валидации: {best_mlp_accuracy:.4f}")


Запуск генетического алгоритма для MLP...

--- Поколение 1/3 ---
Оцениваем особь 1/5: {'learning_rate': 0.003724794591779384, 'num_layers': 3, 'num_neurons': 254, 'activation': 'relu'}


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Новый лучший результат: 0.2222 с параметрами: {'learning_rate': 0.003724794591779384, 'num_layers': 3, 'num_neurons': 254, 'activation': 'relu'}
Оцениваем особь 2/5: {'learning_rate': 0.007597002699807848, 'num_layers': 3, 'num_neurons': 210, 'activation': 'tanh'}
Оцениваем особь 3/5: {'learning_rate': 0.004126137245441604, 'num_layers': 1, 'num_neurons': 238, 'activation': 'tanh'}
Оцениваем особь 4/5: {'learning_rate': 0.0039994619712484195, 'num_layers': 2, 'num_neurons': 155, 'activation': 'tanh'}
Оцениваем особь 5/5: {'learning_rate': 0.0011607445088542003, 'num_layers': 1, 'num_neurons': 66, 'activation': 'relu'}
Новый лучший результат: 0.2826 с параметрами: {'learning_rate': 0.0011607445088542003, 'num_layers': 1, 'num_neurons': 66, 'activation': 'relu'}

--- Поколение 2/3 ---
Оцениваем особь 1/5: {'learning_rate': 0.0011607445088542003, 'num_layers': 1, 'num_neurons': 66, 'activation': 'relu'}
Оцениваем особь 2/5: {'learning_rate': 0.0011607445088542003, 'num_layers': 1, 'num_ne

### 4. Генетический алгоритм для оптимизации CNN

In [ ]:
# Определение пространства поиска гиперпараметров для CNN
cnn_hyperparameter_space = {
    'learning_rate': (0.0001, 0.01), # Диапазон для learning_rate
    'num_conv_layers': (1, 3),      # Количество сверточных слоев
    'filters_per_layer': [
        [16, 32, 64], # Вариант 1: 16-32-64 фильтров
        [32, 64, 128], # Вариант 2: 32-64-128 фильтров
        [8, 16, 32] # Вариант 3: 8-16-32 фильтров
    ],
    'kernel_size': (3, 5),          # Размер ядра свертки (квадратное)
    'num_dense_layers': (1, 2),     # Количество полносвязных слоев после сверточных
    'num_dense_neurons': (64, 128), # Количество нейронов в полносвязных слоях
    'activation': ['relu', 'tanh']  # Функции активации
}

# Функция инициализации особи для CNN
def initialize_cnn_individual():
    lr = random.uniform(cnn_hyperparameter_space['learning_rate'][0],
                        cnn_hyperparameter_space['learning_rate'][1])
    num_conv_layers = random.randint(cnn_hyperparameter_space['num_conv_layers'][0],
                                     cnn_hyperparameter_space['num_conv_layers'][1])
    # Выбираем подсписок фильтров, соответствующий num_conv_layers
    filters_options = [f for f in cnn_hyperparameter_space['filters_per_layer'] if len(f) >= num_conv_layers]
    filters = random.choice(filters_options)[:num_conv_layers]

    kernel_size_val = random.choice(list(range(cnn_hyperparameter_space['kernel_size'][0], cnn_hyperparameter_space['kernel_size'][1] + 1)))
    num_dense_layers = random.randint(cnn_hyperparameter_space['num_dense_layers'][0],
                                      cnn_hyperparameter_space['num_dense_layers'][1])
    num_dense_neurons = random.randint(cnn_hyperparameter_space['num_dense_neurons'][0],
                                       cnn_hyperparameter_space['num_dense_neurons'][1])
    activation = random.choice(cnn_hyperparameter_space['activation'])

    return {
        'learning_rate': lr,
        'num_conv_layers': num_conv_layers,
        'filters_per_layer': filters,
        'kernel_size': (kernel_size_val, kernel_size_val),
        'num_dense_layers': num_dense_layers,
        'num_dense_neurons': num_dense_neurons,
        'activation': activation
    }

# Функция приспособленности для CNN (оценка модели)
def fitness_function_cnn(individual, X_train, y_train, X_val, y_val, input_shape, num_classes, epochs=5):
    hyperparameters = individual
    model = build_cnn_model(input_shape, num_classes, hyperparameters)

    # Обучение модели
    history = model.fit(X_train, y_train, epochs=epochs, validation_data=(X_val, y_val), verbose=0)

    # Возвращаем максимальную точность на валидационном наборе
    return max(history.history['val_accuracy'])


def crossover_cnn(parent1, parent2):
    child1 = parent1.copy()
    child2 = parent2.copy()

    for key in parent1.keys():
        if random.random() < 0.5: # 50% шанс обмена
            child1[key], child2[key] = parent2[key], parent1[key]

    # Особое скрещивание для filters_per_layer
    if random.random() < 0.5: # Шанс обмена списками фильтров
        child1['filters_per_layer'], child2['filters_per_layer'] = parent2['filters_per_layer'], parent1['filters_per_layer']

    # Если num_conv_layers изменилось, убедиться, что filters_per_layer соответствует длине
    if len(child1['filters_per_layer']) > child1['num_conv_layers']:
        child1['filters_per_layer'] = child1['filters_per_layer'][:child1['num_conv_layers']]
    elif len(child1['filters_per_layer']) < child1['num_conv_layers']:
        # Добавить случайные фильтры или использовать из другого родителя
        # Для простоты, переинициализируем этот параметр
        filters_options = [f for f in cnn_hyperparameter_space['filters_per_layer'] if len(f) >= child1['num_conv_layers']]
        if filters_options:
            child1['filters_per_layer'] = random.choice(filters_options)[:child1['num_conv_layers']]
        else:
            child1['filters_per_layer'] = [random.randint(8, 64) for _ in range(child1['num_conv_layers'])]

    if len(child2['filters_per_layer']) > child2['num_conv_layers']:
        child2['filters_per_layer'] = child2['filters_per_layer'][:child2['num_conv_layers']]
    elif len(child2['filters_per_layer']) < child2['num_conv_layers']:
        filters_options = [f for f in cnn_hyperparameter_space['filters_per_layer'] if len(f) >= child2['num_conv_layers']]
        if filters_options:
            child2['filters_per_layer'] = random.choice(filters_options)[:child2['num_conv_layers']]
        else:
            child2['filters_per_layer'] = [random.randint(8, 64) for _ in range(child2['num_conv_layers'])]

    return child1, child2

def mutate_cnn(individual, mutation_rate):
    mutated_individual = individual.copy()
    for key, val_range in cnn_hyperparameter_space.items():
        if random.random() < mutation_rate:
            if key == 'learning_rate':
                mutated_individual[key] = random.uniform(val_range[0], val_range[1])
            elif key == 'num_conv_layers' or key == 'num_dense_layers':
                mutated_individual[key] = random.randint(val_range[0], val_range[1])
            elif key == 'num_dense_neurons':
                mutated_individual[key] = random.randint(val_range[0], val_range[1])
            elif key == 'activation':
                mutated_individual[key] = random.choice(val_range)
            elif key == 'kernel_size':
                kernel_size_val = random.choice(list(range(val_range[0], val_range[1] + 1)))
                mutated_individual[key] = (kernel_size_val, kernel_size_val)
            elif key == 'filters_per_layer':
                num_conv_layers = mutated_individual['num_conv_layers']
                # Пересоздаем список фильтров, чтобы он соответствовал новому num_conv_layers
                filters_options = [f for f in cnn_hyperparameter_space['filters_per_layer'] if len(f) >= num_conv_layers]
                if filters_options:
                    mutated_individual[key] = random.choice(filters_options)[:num_conv_layers]
                else:
                    mutated_individual[key] = [random.randint(8, 64) for _ in range(num_conv_layers)]

    # После мутации num_conv_layers, убедиться, что filters_per_layer соответствует длине
    if len(mutated_individual['filters_per_layer']) != mutated_individual['num_conv_layers']:
        num_conv_layers = mutated_individual['num_conv_layers']
        filters_options = [f for f in cnn_hyperparameter_space['filters_per_layer'] if len(f) >= num_conv_layers]
        if filters_options:
            mutated_individual['filters_per_layer'] = random.choice(filters_options)[:num_conv_layers]
        else:
            mutated_individual['filters_per_layer'] = [random.randint(8, 64) for _ in range(num_conv_layers)]

    return mutated_individual

# Запуск ГА для CNN
print("\nЗапуск генетического алгоритма для CNN...")
best_cnn_params, best_cnn_accuracy = run_genetic_algorithm(
    initialize_cnn_individual,
    fitness_function_cnn,
    crossover_cnn,
    mutate_cnn,
    population_size,
    num_generations,
    mutation_rate,
    X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
    input_shape=X_train.shape[1:], num_classes=y_train.shape[1]
)

print("\n--- Результаты оптимизации CNN ---")
print(f"Лучшие гиперпараметры CNN: {best_cnn_params}")
print(f"Лучшая точность CNN на валидации: {best_cnn_accuracy:.4f}")



Запуск генетического алгоритма для CNN...

--- Поколение 1/3 ---
Оцениваем особь 1/5: {'learning_rate': 0.009395418768352568, 'num_conv_layers': 3, 'filters_per_layer': [8, 16, 32], 'kernel_size': (3, 3), 'num_dense_layers': 2, 'num_dense_neurons': 64, 'activation': 'relu'}


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Новый лучший результат: 0.1030 с параметрами: {'learning_rate': 0.009395418768352568, 'num_conv_layers': 3, 'filters_per_layer': [8, 16, 32], 'kernel_size': (3, 3), 'num_dense_layers': 2, 'num_dense_neurons': 64, 'activation': 'relu'}
Оцениваем особь 2/5: {'learning_rate': 0.0034420747581564003, 'num_conv_layers': 2, 'filters_per_layer': [32, 64], 'kernel_size': (3, 3), 'num_dense_layers': 2, 'num_dense_neurons': 101, 'activation': 'tanh'}
Новый лучший результат: 0.3776 с параметрами: {'learning_rate': 0.0034420747581564003, 'num_conv_layers': 2, 'filters_per_layer': [32, 64], 'kernel_size': (3, 3), 'num_dense_layers': 2, 'num_dense_neurons': 101, 'activation': 'tanh'}
Оцениваем особь 3/5: {'learning_rate': 0.005700824788422797, 'num_conv_layers': 2, 'filters_per_layer': [8, 16], 'kernel_size': (3, 3), 'num_dense_layers': 2, 'num_dense_neurons': 70, 'activation': 'relu'}
Новый лучший результат: 0.4842 с параметрами: {'learning_rate': 0.005700824788422797, 'num_conv_layers': 2, 'filters

### 5. Сравнение результатов и выбор лучшей модели

In [ ]:
print(f"MLP: Лучшая точность на валидации: {best_mlp_accuracy:.4f} с параметрами: {best_mlp_params}")
print(f"CNN: Лучшая точность на валидации: {best_cnn_accuracy:.4f} с параметрами: {best_cnn_params}")

if best_mlp_accuracy > best_cnn_accuracy:
    print("\nЛучшая модель: MLP")
    best_model_type = "MLP"
    final_best_params = best_mlp_params
    final_best_accuracy = best_mlp_accuracy
elif best_cnn_accuracy > best_mlp_accuracy:
    print("\nЛучшая модель: CNN")
    best_model_type = "CNN"
    final_best_params = best_cnn_params
    final_best_accuracy = best_cnn_accuracy
else:
    print("\nМодели MLP и CNN показали одинаковую лучшую точность.")
    best_model_type = "Обе"
    final_best_params = "N/A"
    final_best_accuracy = best_mlp_accuracy

print(f"\nИтак, на основе генетического алгоритма, лучшей моделью оказалась {best_model_type} с точностью {final_best_accuracy:.4f}.")


MLP: Лучшая точность на валидации: 0.2882 с параметрами: {'learning_rate': 0.0011607445088542003, 'num_layers': 1, 'num_neurons': 66, 'activation': 'relu'}
CNN: Лучшая точность на валидации: 0.5833 с параметрами: {'learning_rate': 0.0017444071218576894, 'num_conv_layers': 1, 'filters_per_layer': [16], 'kernel_size': (3, 3), 'num_dense_layers': 2, 'num_dense_neurons': 100, 'activation': 'relu'}

Лучшая модель: CNN

Итак, на основе генетического алгоритма, лучшей моделью оказалась CNN с точностью 0.5833.


В итоге череды обучений получили лучшую сеть с точностью 0.5833. Стоит отметить, что выполнение всего блокнота заняло 2 часа 20 минут. Для такой простой задачи обучить сеть на точность 0.95+ можно за 5 минут, просто выбрав гиперпараметры наугад. Для таких задач применение генетических алгоритмов актуально только в качестве демонстрации их возможностей. В данном блокноте было обучено 15 MLP и 15 CNN моделей, а так же выполнялась логика генетических алгоритмов.